<a href="https://colab.research.google.com/github/novikovamaria137-png/mtuci-llm-course/blob/main/lesson-2.4/practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# Практика 2.4. Вызов функций и первый агент

**Модуль 2 · Урок 4 · 110 минут**

В уроке 2.3 модель возвращала данные. Сегодня она будет возвращать **намерение**: какую функцию вызвать и с какими аргументами. Выполнять будет ваша программа — и отсюда всё остальное содержание урока.

---

### Что вы сделаете

| Шаг | Что делаем | Время | Нужен ключ |
|---|---|---|---|
| 0 | Восстановим каркас уроков 2.1–2.3 | 10 мин | нет |
| 1 | Опишем инструмент схемой и соберём реестр | 20 мин | нет |
| 2 | Проверим белый список: модель называет несуществующее | 10 мин | нет |
| 3 | Проверим аргументы, включая выдуманные моделью | 20 мин | нет |
| 4 | Разделим читающие и изменяющие операции | 15 мин | нет |
| 5 | Соберём цикл агента | 20 мин | нет |
| 6 | Проверим три способа никогда не остановиться | 15 мин | нет |
| 7 | Дадим инструменты настоящей модели | 10 мин | да |

> **Ключевая мысль урока — дословно из документации GigaChat:** «Модели не исполняют функции самостоятельно, а принимают решения о работе с ними». Модель возвращает имя функции и аргументы. Выполняете вы.

---
## Шаг 0. Каркас

Четыре модуля из прошлых уроков — без изменений.

*Статус ячейки: проверено запуском.*

In [ ]:
REQUIREMENTS = ["openai==2.51.0", "python-dotenv==1.2.2"]

import importlib.util, subprocess, sys
from pathlib import Path

def ensure(spec):
    name = spec.split("==")[0].replace("-", "_")
    if importlib.util.find_spec(name) is None:
        print(f"  устанавливаю {spec} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=False)
    else:
        print(f"  {name}: уже установлен")

print("Зависимости:")
for spec in REQUIREMENTS:
    ensure(spec)

ROOT = Path("llm-project")
(ROOT / "llmcourse").mkdir(parents=True, exist_ok=True)
(ROOT / "llmcourse" / "__init__.py").write_text("", encoding="utf-8")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

print("\nПроект:", ROOT.resolve())

In [ ]:
%%writefile llm-project/llmcourse/config.py
"""Единая точка настройки. Урок 2.1.

Ключи НИКОГДА не пишутся в коде. Порядок поиска:
  1. Colab Secrets  (значок ключа слева в Colab)
  2. переменные окружения
  3. файл .env рядом с проектом
Если ключа нет — включается автономный режим на заглушке.
"""
import os
from pathlib import Path

ENV_KEYS = ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL")


def _from_colab(name):
    try:
        from google.colab import userdata          # есть только в Colab
        return userdata.get(name)
    except Exception:
        return None


def _from_dotenv(name, path=".env"):
    p = Path(path)
    if not p.exists():
        return None
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        if k.strip() == name:
            return v.strip().strip('"').strip("'")
    return None


def get(name, default=None):
    """Достаёт значение из Colab Secrets, окружения или .env."""
    return _from_colab(name) or os.environ.get(name) or _from_dotenv(name) or default


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def settings():
    """Возвращает конфигурацию и признак автономного режима."""
    cfg = {k: get(k) for k in ENV_KEYS}
    cfg["OFFLINE"] = not bool(cfg["LLM_API_KEY"])
    cfg["MODEL"] = cfg["LLM_MODEL"] or "demo-model"
    return cfg


def describe():
    """Человекочитаемый отчёт об окружении."""
    cfg = settings()
    where = "Google Colab" if in_colab() else "локальная среда"
    return "\n".join([
        f"Среда:            {where}",
        f"Модель:           {cfg['MODEL']}",
        f"Базовый адрес:    {cfg['LLM_BASE_URL'] or 'не задан'}",
        f"Ключ:             {'найден' if not cfg['OFFLINE'] else 'НЕ найден'}",
        f"Режим:            {'автономный (заглушка)' if cfg['OFFLINE'] else 'обращение к API'}",
    ])


In [ ]:
%%writefile llm-project/llmcourse/client.py
"""Клиент для работы с языковой моделью. Уроки 2.1–2.2.

Написан на OpenAI-совместимый интерфейс: работает с российскими API
и с локальными рантаймами. Смена поставщика — правка .env, не кода.
Без ключа работает в автономном режиме на заглушке.
"""
import time, random, hashlib
from dataclasses import dataclass
from . import config


@dataclass
class Usage:
    """Накопительный счётчик расхода."""
    calls: int = 0
    tokens_in: int = 0
    tokens_out: int = 0
    price_in: float = 0.0     # рублей за 1000 входных токенов
    price_out: float = 0.0    # рублей за 1000 выходных

    @property
    def cost(self):
        return (self.tokens_in / 1000 * self.price_in +
                self.tokens_out / 1000 * self.price_out)

    def report(self):
        return (f"обращений: {self.calls}   "
                f"токенов: {self.tokens_in} вход / {self.tokens_out} выход   "
                f"стоимость: {self.cost:.4f} руб.")


def approx_tokens(text):
    """Грубая ОЦЕНКА числа токенов до вызова API.

    Это оценка, а не замер: точное число даёт токенизатор конкретной
    модели (см. урок 1.2). Нужна, чтобы прикинуть стоимость заранее.
    """
    return max(1, len(text) // 3)


class LLM:
    def __init__(self, price_in=0.0, price_out=0.0, max_retries=4, timeout=60):
        cfg = config.settings()
        self.offline = cfg["OFFLINE"]
        self.model = cfg["MODEL"]
        self.base_url = cfg["LLM_BASE_URL"]
        self.max_retries = max_retries
        self.usage = Usage(price_in=price_in, price_out=price_out)
        self._client = None
        if not self.offline:
            from openai import OpenAI
            self._client = OpenAI(base_url=self.base_url,
                                  api_key=cfg["LLM_API_KEY"],
                                  timeout=timeout)

    def _offline_answer(self, messages):
        """Детерминированный ответ: одинаковый запрос — одинаковый ответ."""
        text = " ".join(m["content"] for m in messages)
        h = hashlib.sha256(text.encode()).hexdigest()[:6]
        return (f"[автономный режим] Ответ-заглушка {h}. "
                f"Получено сообщений: {len(messages)}, символов: {len(text)}. "
                f"Подставьте ключ, чтобы обратиться к модели.")

    @staticmethod
    def _is_retryable(e):
        """Повторяем только то, что имеет шанс пройти со второго раза."""
        if type(e).__name__ in ("RateLimitError", "APITimeoutError",
                                "APIConnectionError", "InternalServerError",
                                "TimeoutError", "ConnectionError"):
            return True
        return getattr(e, "status_code", None) in (408, 429, 500, 502, 503, 504)

    def _with_retry(self, fn):
        """Экспоненциальная задержка со случайной добавкой."""
        for attempt in range(self.max_retries):
            try:
                return fn()
            except Exception as e:
                if not self._is_retryable(e) or attempt == self.max_retries - 1:
                    raise
                time.sleep(0.5 * (2 ** attempt) + random.uniform(0, 0.3))

    def ask(self, prompt, system=None, temperature=0.2, max_tokens=None):
        messages = ([{"role": "system", "content": system}] if system else []) + \
                   [{"role": "user", "content": prompt}]
        self.usage.calls += 1

        if self.offline:
            answer = self._offline_answer(messages)
            self.usage.tokens_in += approx_tokens(" ".join(m["content"] for m in messages))
            self.usage.tokens_out += approx_tokens(answer)
            return answer

        def call():
            kw = dict(model=self.model, messages=messages, temperature=temperature)
            if max_tokens:
                kw["max_tokens"] = max_tokens
            return self._client.chat.completions.create(**kw)

        resp = self._with_retry(call)
        u = getattr(resp, "usage", None)
        if u:
            self.usage.tokens_in += getattr(u, "prompt_tokens", 0)
            self.usage.tokens_out += getattr(u, "completion_tokens", 0)
        return resp.choices[0].message.content


In [ ]:
%%writefile llm-project/llmcourse/prompts.py
"""Промпт как часть программы. Урок 2.2.

Промпт хранится отдельно от кода вызова: у него есть имя, версия и явный
список параметров. Это позволяет менять формулировку, не трогая логику,
и видеть в журнале, какой версией промпта получен ответ.

Почему не str.format. Во-первых, .format умеет обращаться к атрибутам объекта
("{x.__class__}"), и на шаблоне, пришедшем извне, это дыра в безопасности.
Во-вторых, нам нужна подстановка и ничего больше — а всё лишнее в инструменте
рано или поздно кто-нибудь применит.

Правила шаблона те же, что в Python, чтобы знание переносилось:
    {name}  — параметр
    {{      — литеральная открывающая скобка
    }}      — литеральная закрывающая скобка
Одиночная } без пары — ошибка. Это не придирка: чаще всего она означает
незакрытый или неверно записанный параметр.
"""
import re
from dataclasses import dataclass

# Имя параметра — любой допустимый идентификатор, включая кириллицу: Python это
# разрешает. Но в примерах курса имена латинские — так принято, и так их видно
# в чужом коде без сюрпризов с раскладкой.
_NAME = re.compile(r"[^\W\d]\w*")


class PromptError(ValueError):
    """Ошибка в шаблоне промпта или в его заполнении."""


def parse(text):
    """Разбирает шаблон в список кусков: ("lit", строка) или ("field", имя)."""
    out, buf, i, n = [], [], 0, len(text)

    def flush():
        if buf:
            out.append(("lit", "".join(buf)))
            buf.clear()

    while i < n:
        ch = text[i]
        if ch == "{":
            if i + 1 < n and text[i + 1] == "{":
                buf.append("{"); i += 2; continue
            j = text.find("}", i + 1)
            if j == -1:
                raise PromptError("незакрытая « { » в шаблоне")
            name = text[i + 1:j]
            if not _NAME.fullmatch(name):
                raise PromptError(
                    f"недопустимое имя параметра: {{{name}}}. "
                    "Для литеральной скобки используйте {{ и }}")
            flush()
            out.append(("field", name))
            i = j + 1
            continue
        if ch == "}":
            if i + 1 < n and text[i + 1] == "}":
                buf.append("}"); i += 2; continue
            raise PromptError(
                "одиночная « } » в шаблоне. Для литеральной скобки пишите }}")
        buf.append(ch); i += 1
    flush()
    return out


@dataclass(frozen=True)
class Prompt:
    name: str
    version: str
    template: str
    system: str = ""

    @property
    def fields(self):
        """Имена параметров, которые нужно передать при заполнении."""
        parts = parse(self.template) + parse(self.system)
        return sorted({v for kind, v in parts if kind == "field"})

    def render(self, **values):
        """Заполняет шаблон. Молча ничего не проглатывает."""
        need, got = set(self.fields), set(values)
        if need - got:
            raise PromptError(
                f"{self.label()}: не переданы параметры {sorted(need - got)}")
        if got - need:
            raise PromptError(
                f"{self.label()}: лишние параметры {sorted(got - need)}. "
                f"Ожидались {self.fields}. Опечатка в имени — частая причина")

        def fill(text):
            return "".join(
                v if kind == "lit" else str(values[v]) for kind, v in parse(text))

        return {"system": fill(self.system), "user": fill(self.template)}

    def label(self):
        """Метка для журнала: по ней потом видно, чем получен ответ."""
        return f"{self.name}@{self.version}"


class Registry:
    """Все промпты проекта в одном месте."""

    def __init__(self):
        self._items = {}

    def add(self, prompt):
        old = self._items.get(prompt.name)
        if old is not None and old.version == prompt.version:
            raise PromptError(
                f"промпт {prompt.name} версии {prompt.version} уже зарегистрирован. "
                "Меняете формулировку — поднимите версию")
        self._items[prompt.name] = prompt
        return prompt

    def get(self, name):
        if name not in self._items:
            raise PromptError(f"промпт {name} не найден. Есть: {sorted(self._items)}")
        return self._items[name]

    def names(self):
        return sorted(self._items)


In [ ]:
%%writefile llm-project/llmcourse/structured.py
"""Структурированный вывод. Урок 2.3.

Модель возвращает текст. Даже когда вы просите JSON и даже когда поставщик
поддерживает строгую схему, на вход разбора приходит строка — и в ней, кроме
самого JSON, регулярно оказывается лишнее:

    ```json ... ```      обрамление markdown
    Вот результат: {...}  вежливая преамбула
    {...} Надеюсь, помог  постамбула
    {"a": 1,              обрыв по max_tokens

Поэтому разбор должен быть устойчивым, а проверка — строгой. Модуль делает
ровно две вещи: достаёт JSON из шумного текста и проверяет его на соответствие
ожидаемой форме.
"""
import json
import re

FENCE = re.compile(r"```[a-zA-Z]*\s*\n?(.*?)```", re.S)


class StructureError(ValueError):
    """Ответ модели не удалось привести к ожидаемой форме."""


def strip_fences(text):
    """Снимает обрамление ```...```, если оно есть."""
    m = FENCE.search(text)
    return m.group(1) if m else text


def find_json(text):
    """Находит первый сбалансированный JSON-объект или массив.

    Сканер посимвольный, а не регулярное выражение: скобки встречаются внутри
    строк, и регулярное выражение на этом ломается. Пример, на котором ломаются
    почти все самодельные варианты:

        {"note": "закрывающая } внутри строки"}
    """
    src = strip_fences(text)
    start = None
    for i, ch in enumerate(src):
        if ch in "{[":
            start = i
            break
    if start is None:
        raise StructureError(
            "в ответе модели нет ни { ни [ — JSON отсутствует. "
            f"Начало ответа: {src[:80]!r}")

    opener = src[start]
    closer = "}" if opener == "{" else "]"
    depth, in_str, esc = 0, False, False

    for i in range(start, len(src)):
        ch = src[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == opener:
            depth += 1
        elif ch == closer:
            depth -= 1
            if depth == 0:
                return src[start:i + 1]

    raise StructureError(
        "JSON начался, но не закончился. Самая частая причина — ответ обрезан "
        "по max_tokens: проверьте finish_reason, при значении 'length' "
        "поднимите потолок")


def parse(text):
    """Достаёт JSON из ответа модели и разбирает его."""
    raw = find_json(text)
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        raise StructureError(
            f"JSON найден, но не разобран: {e.msg} (позиция {e.pos}). "
            f"Фрагмент: {raw[max(0, e.pos - 30):e.pos + 30]!r}")


# ── проверка формы ────────────────────────────────────────────────────
def check(data, required=(), types=None):
    """Проверяет разобранные данные на соответствие ожиданиям.

    required — имена обязательных полей;
    types    — словарь «поле: тип» для полей, тип которых важен.

    Возвращает список претензий. Пустой список означает, что всё в порядке.
    Список, а не первая ошибка: за один проход видно всё, что не так, —
    это заметно быстрее при отладке промпта.
    """
    problems = []
    if not isinstance(data, dict):
        return [f"ожидался объект, получен {type(data).__name__}"]
    for f in required:
        if f not in data:
            problems.append(f"нет обязательного поля {f!r}")
        elif data[f] is None:
            problems.append(f"поле {f!r} пустое (null)")
    for f, t in (types or {}).items():
        if f in data and data[f] is not None and not isinstance(data[f], t):
            problems.append(
                f"поле {f!r}: ожидался {t.__name__}, получен {type(data[f]).__name__}")
    return problems


def parse_checked(text, required=(), types=None):
    """Разбор с проверкой. Бросает StructureError со всеми претензиями сразу."""
    data = parse(text)
    problems = check(data, required, types)
    if problems:
        raise StructureError("ответ не соответствует ожиданиям: " + "; ".join(problems))
    return data


# ── цикл починки ──────────────────────────────────────────────────────
def ask_structured(ask, prompt, required=(), types=None, attempts=3, log=None):
    """Запрашивает структурированный ответ, при неудаче показывает модели её ошибку.

    ask — функция, принимающая текст и возвращающая ответ модели.

    Ограничение числа попыток обязательно. Без него неудачный промпт
    превращается в бесконечный платный цикл — ровно та ошибка, которую
    разбирали в уроке 2.1.
    """
    if attempts < 1:
        raise ValueError("attempts должно быть не меньше 1")
    log = log if log is not None else []
    text = prompt

    for n in range(1, attempts + 1):
        answer = ask(text)
        try:
            data = parse_checked(answer, required, types)
            log.append({"attempt": n, "ok": True})
            return data
        except StructureError as e:
            log.append({"attempt": n, "ok": False, "error": str(e)})
            if n == attempts:
                raise StructureError(
                    f"не удалось получить корректный ответ за {attempts} попыт(ки). "
                    f"Последняя ошибка: {e}")
            text = (
                f"{prompt}\n\n"
                f"Предыдущий ответ не подошёл: {e}\n"
                "Верни только JSON-объект, без пояснений и без обрамления markdown."
            )
    raise AssertionError("недостижимо")


In [ ]:
import importlib
importlib.invalidate_caches()
from llmcourse import config, client, prompts, structured
for m in (config, client, prompts, structured):
    importlib.reload(m)

print(config.describe())
print("\nКаркас уроков 2.1-2.3 на месте.")

---
## Шаг 1. Инструмент — это схема плюс функция

Как модель узнаёт, что она может вызвать? Вы передаёте ей список описаний. Каждое описание — почти то же самое, что схема из урока 2.3.

| Поле | Что означает | Кто им пользуется |
|---|---|---|
| `name` | Имя функции | Модель называет его в ответе |
| `description` | Что функция делает | **Модель выбирает инструмент по описанию** |
| `parameters` | JSON-схема аргументов | Модель по ней генерирует аргументы |
| `fn` | Собственно функция | Только ваш код. Модель её не видит |
| `writes` | Меняет ли состояние мира | Только ваш код. Об этом — шаг 4 |

**Описание важнее, чем кажется.** Модель выбирает инструмент, читая описание, и больше ей опереться не на что. Описание «Возвращает данные» гарантирует, что выбор будет случайным.

### Что возвращает модель

Из документации GigaChat, пример ответа при вызове функции:

```json
"function_call": {
    "name": "weather_forecast",
    "arguments": {"format": "celsius", "location": "Манжерок", "num_days": 10}
},
"finish_reason": "function_call"
```

Обратите внимание на `finish_reason`. Это то самое поле из урока 2.1: здесь оно говорит, что модель не закончила ответ, а просит вызвать функцию.

*Статус ячейки: проверено запуском.*

In [ ]:
%%writefile llm-project/llmcourse/tools.py
"""Вызов функций и цикл агента. Урок 2.4.

Ключевая мысль урока, дословно из документации GigaChat:
«Модели не исполняют функции самостоятельно, а принимают решения о работе
с ними, опираясь на имеющиеся знания, текущий разговор и описание функций
из запроса».

То есть модель возвращает не результат, а намерение: имя функции и аргументы.
Выполняет — ваша программа. Отсюда всё остальное содержание модуля: раз
выполняете вы, то вы и отвечаете за то, что именно будет выполнено.
"""
import json
import re
from dataclasses import dataclass, field

NAME_RE = re.compile(r"[a-zA-Z_][a-zA-Z0-9_]*")


class ToolError(Exception):
    """Проблема с инструментом: его нет, аргументы не те, вызов запрещён."""


@dataclass
class Tool:
    """Описание инструмента, доступного модели.

    parameters — JSON-схема аргументов, тот же формат, что в уроке 2.3.
    writes — признак того, что инструмент меняет состояние мира. Отделён
    от читающих намеренно: неверный читающий вызов стоит времени,
    неверный изменяющий может стоить гораздо большего.
    """
    name: str
    description: str
    parameters: dict
    fn: callable
    writes: bool = False

    def __post_init__(self):
        if not NAME_RE.fullmatch(self.name):
            raise ToolError(
                f"недопустимое имя инструмента {self.name!r}: "
                "только латиница, цифры и подчёркивание")
        if not self.description.strip():
            raise ToolError(
                f"{self.name}: пустое описание. Модель выбирает инструмент "
                "по описанию — без него выбор будет случайным")
        if self.parameters.get("type") != "object":
            raise ToolError(f"{self.name}: parameters должен быть объектом JSON-схемы")

    def describe(self):
        """Описание в том виде, в каком его ждёт API."""
        return {"name": self.name, "description": self.description,
                "parameters": self.parameters}


class Registry:
    """Белый список инструментов.

    Модель может назвать что угодно, в том числе несуществующее. Реестр —
    единственное место, где решается, будет ли что-то выполнено.
    """

    def __init__(self, tools=()):
        self._tools = {}
        for t in tools:
            self.add(t)

    def add(self, tool):
        if tool.name in self._tools:
            raise ToolError(f"инструмент {tool.name} уже зарегистрирован")
        self._tools[tool.name] = tool
        return tool

    def names(self):
        return sorted(self._tools)

    def describe(self):
        """Массив описаний для передачи в запрос."""
        return [self._tools[n].describe() for n in self.names()]

    # ── проверка аргументов ────────────────────────────────────────
    _TYPES = {"string": str, "integer": int, "number": (int, float),
              "boolean": bool, "array": list, "object": dict}

    def _validate(self, tool, args):
        problems = []
        if not isinstance(args, dict):
            return [f"аргументы должны быть объектом, получен {type(args).__name__}"]
        props = tool.parameters.get("properties", {})
        required = tool.parameters.get("required", [])

        for name in required:
            if name not in args:
                problems.append(f"нет обязательного аргумента {name!r}")

        for name, value in args.items():
            if name not in props:
                problems.append(
                    f"аргумент {name!r} не описан в схеме. "
                    "Модель могла его выдумать")
                continue
            spec = props[name]
            expected = self._TYPES.get(spec.get("type"))
            if expected and not isinstance(value, expected):
                problems.append(
                    f"аргумент {name!r}: ожидался {spec['type']}, "
                    f"получен {type(value).__name__}")
                continue
            if "enum" in spec and value not in spec["enum"]:
                problems.append(
                    f"аргумент {name!r}: значение {value!r} не входит "
                    f"в допустимые {spec['enum']}")
        return problems

    # ── вызов ──────────────────────────────────────────────────────
    def call(self, name, args, allow_writes=False):
        """Выполняет инструмент после всех проверок.

        allow_writes выключен по умолчанию. Изменяющие операции требуют
        явного разрешения на стороне вызывающего кода — чтобы включение
        такой возможности было осознанным решением, а не умолчанием.
        """
        tool = self._tools.get(name)
        if tool is None:
            raise ToolError(
                f"инструмент {name!r} не зарегистрирован. "
                f"Доступны: {self.names()}")
        if tool.writes and not allow_writes:
            raise ToolError(
                f"инструмент {name!r} изменяет состояние и запрещён "
                "в этом режиме. Разрешите явно, если это то, что нужно")
        problems = self._validate(tool, args)
        if problems:
            raise ToolError(f"{name}: " + "; ".join(problems))
        return tool.fn(**args)


# ── цикл агента ────────────────────────────────────────────────────
@dataclass
class Step:
    n: int
    kind: str            # "call" | "answer" | "error"
    tool: str = ""
    args: dict = field(default_factory=dict)
    result: object = None
    error: str = ""


def run_agent(decide, registry, task, max_steps=5, allow_writes=False):
    """Цикл агента.

    decide — функция, которой передаётся задача и журнал уже сделанного;
    она возвращает либо {"tool": имя, "args": {...}}, либо {"answer": текст}.
    В настоящем приложении её роль играет модель.

    Агент — это не магия, а цикл с ограничением. Ограничений здесь три,
    и каждое закрывает свой способ никогда не остановиться:
      max_steps          — общий предел числа шагов;
      повтор вызова      — тот же инструмент с теми же аргументами подряд;
      ошибка инструмента — возвращается модели, но шаг всё равно потрачен.
    """
    if max_steps < 1:
        raise ValueError("max_steps должно быть не меньше 1")

    trace, last_call = [], None
    for n in range(1, max_steps + 1):
        decision = decide(task, trace)

        if "answer" in decision:
            trace.append(Step(n=n, kind="answer", result=decision["answer"]))
            return {"answer": decision["answer"], "trace": trace, "stopped": "ответ"}

        name = decision.get("tool")
        args = decision.get("args", {})

        signature = (name, json.dumps(args, sort_keys=True, ensure_ascii=False))
        if signature == last_call:
            trace.append(Step(n=n, kind="error", tool=name, args=args,
                              error="повтор того же вызова с теми же аргументами"))
            return {"answer": None, "trace": trace, "stopped": "зацикливание"}
        last_call = signature

        try:
            result = registry.call(name, args, allow_writes=allow_writes)
            trace.append(Step(n=n, kind="call", tool=name, args=args, result=result))
        except ToolError as e:
            trace.append(Step(n=n, kind="error", tool=name, args=args, error=str(e)))

    return {"answer": None, "trace": trace, "stopped": "исчерпан лимит шагов"}


In [ ]:
import importlib
importlib.invalidate_caches()
import llmcourse.tools
importlib.reload(llmcourse.tools)
from llmcourse.tools import Tool, Registry, ToolError, run_agent

# ── два инструмента: один читает, другой меняет ──────────────────
def weather(location, num_days, format="celsius"):
    """Заглушка вместо настоящего сервиса погоды."""
    return {"location": location, "days": num_days, "temp": 17, "unit": format}

def send_mail(to, text):
    """Заглушка вместо настоящей отправки."""
    return {"sent_to": to, "length": len(text)}

WEATHER = Tool(
    name="weather_forecast",
    description="Возвращает прогноз температуры для города на заданное число дней",
    parameters={
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "Название города"},
            "num_days": {"type": "integer", "description": "На сколько дней вперёд"},
            "format":   {"type": "string", "enum": ["celsius", "fahrenheit"],
                         "description": "Единицы измерения температуры"},
        },
        "required": ["location", "num_days"],
    },
    fn=weather,
)

MAIL = Tool(
    name="send_mail",
    description="Отправляет письмо указанному адресату",
    parameters={
        "type": "object",
        "properties": {"to": {"type": "string", "description": "Адрес получателя"},
                       "text": {"type": "string", "description": "Текст письма"}},
        "required": ["to", "text"],
    },
    fn=send_mail,
    writes=True,          # ← меняет состояние мира
)

reg = Registry([WEATHER, MAIL])
print("Зарегистрировано:", reg.names())
print()
import json as _j
print("Так описания уходят в запрос:")
print(_j.dumps(reg.describe()[1], ensure_ascii=False, indent=2)[:420], "...")

In [ ]:
# Битые описания не проходят регистрацию.
bad = [
    ("имя с пробелом",  dict(name="плохое имя", description="д", parameters={"type": "object"})),
    ("пустое описание", dict(name="ok_name", description="   ", parameters={"type": "object"})),
    ("не объект",       dict(name="ok_name", description="д", parameters={"type": "string"})),
]
for label, kw in bad:
    try:
        Tool(fn=lambda: None, **kw)
        raise SystemExit("проверка провалена: " + label)
    except ToolError as e:
        print(f"  [ok] {label}: {str(e)[:78]}")

try:
    reg.add(WEATHER)
except ToolError as e:
    print(f"  [ok] повторная регистрация: {e}")

---
## Шаг 2. Белый список

Модель может назвать что угодно — в том числе инструмент, которого не существует. Это не редкость и не признак плохой модели: она видит только описания и работает с текстом.

Реестр — **единственное место**, где решается, будет ли что-то выполнено. Отсюда правило: никогда не вызывайте функцию по имени, пришедшему от модели, без проверки по списку.

*Статус ячейки: проверено запуском.*

In [ ]:
for name in ("delete_everything", "weather", "WeatherForecast"):
    try:
        reg.call(name, {})
        raise SystemExit("проверка провалена: " + name)
    except ToolError as e:
        print(f"  [ok] {name!r} отклонён")
        print(f"       {e}")
        print()

print("Обратите внимание: 'weather' и 'WeatherForecast' — правдоподобные имена.")
print("Модель могла сократить или изменить регистр. Совпадение должно быть точным.")

---
## Шаг 3. Аргументы

Имя проверили. Теперь аргументы — и здесь есть случай, которого не было в уроке 2.3.

Модель может **выдумать аргумент**, которого нет в схеме. Например, увидев в описании слово «единицы», добавить `units` вместо `format`. Если вы передаёте аргументы в функцию напрямую, это даст `TypeError` в лучшем случае — а в худшем функция примет лишний аргумент и поведёт себя не так, как вы ожидали.

Проверяем четыре вещи:

| Что проверяем | Пример нарушения |
|---|---|
| Обязательные аргументы на месте | нет `num_days` |
| Типы соответствуют схеме | `num_days` пришёл строкой «десять» |
| Значения из `enum` допустимы | `format` = `"kelvin"` |
| Нет аргументов вне схемы | модель добавила `units` |

*Статус ячейки: проверено запуском.*

In [ ]:
print("Корректный вызов:")
print("  ", reg.call("weather_forecast", {"location": "Манжерок", "num_days": 10}))
print("   (format не передан — подставилось умолчание функции)")
print()

wrong = [
    ("нет обязательного",  {"location": "Манжерок"}),
    ("неверный тип",       {"location": "Манжерок", "num_days": "десять"}),
    ("значение вне enum",  {"location": "Манжерок", "num_days": 3, "format": "kelvin"}),
    ("выдуманный аргумент",{"location": "Манжерок", "num_days": 3, "units": "celsius"}),
    ("сразу несколько",    {"num_days": "три", "units": "c"}),
    ("вообще не объект",   ["Манжерок", 10]),
]
for label, args in wrong:
    try:
        reg.call("weather_forecast", args)
        raise SystemExit("проверка провалена: " + label)
    except ToolError as e:
        print(f"  [ok] {label}:")
        print(f"       {e}")

---
## Шаг 4. Читающие и изменяющие операции

Это главный слайд урока по последствиям, поэтому отдельный шаг.

Неверный **читающий** вызов стоит времени: вы получили не ту погоду, посмотрели, переспросили. Неверный **изменяющий** вызов может стоить существенно большего: письмо ушло, запись удалена, платёж отправлен.

Разница не в коде, а в том, что происходит с миром снаружи. Поэтому в нашем реестре изменяющие инструменты помечены признаком `writes` и **по умолчанию запрещены**. Чтобы их разрешить, нужно явно передать `allow_writes=True`.

**Почему умолчание именно такое.** Умолчание — это то, что происходит, когда никто ни о чём не подумал. Значит, безопасным должно быть именно оно.

*Статус ячейки: проверено запуском.*

In [ ]:
try:
    reg.call("send_mail", {"to": "rector@mtuci.ru", "text": "Здравствуйте"})
    raise SystemExit("проверка провалена")
except ToolError as e:
    print("[ok] по умолчанию запрещено:")
    print("    ", e)

print()
r = reg.call("send_mail", {"to": "rector@mtuci.ru", "text": "Здравствуйте"},
             allow_writes=True)
print("[ok] с явным разрешением выполнено:", r)

print()
print("Разница между двумя строками выше — одна именованная переменная.")
print("Именно поэтому она названа так, что её нельзя передать случайно.")

---
## Шаг 5. Цикл агента

Слово «агент» звучит внушительно. Внутри — цикл:

```
повторять, пока не кончились шаги:
    спросить модель, что делать дальше
    если она даёт ответ  → вернуть его и выйти
    если просит вызов    → проверить, выполнить, записать результат
```

И всё. Никакой другой магии там нет.

Ниже вместо модели работает заглушка `decide` — обычная функция, принимающая задачу и журнал уже сделанного. Это позволяет проверить логику цикла без ключа и без случайности.

*Статус ячейки: проверено запуском.*

In [ ]:
def decide_normal(task, trace):
    """Заглушка вместо модели: сначала запрашивает погоду, потом отвечает."""
    if not trace:
        return {"tool": "weather_forecast",
                "args": {"location": "Манжерок", "num_days": 10}}
    return {"answer": "В Манжероке около 17 градусов в ближайшие 10 дней."}

res = run_agent(decide_normal, reg, "Какая погода в Манжероке на 10 дней?")

print("Остановился:", res["stopped"])
print("Ответ:      ", res["answer"])
print()
print("Журнал:")
for s in res["trace"]:
    if s.kind == "call":
        print(f"  {s.n}. вызов {s.tool}({s.args}) -> {s.result}")
    elif s.kind == "answer":
        print(f"  {s.n}. ответ: {s.result}")
    else:
        print(f"  {s.n}. ошибка {s.tool}: {s.error}")

**Журнал — не украшение.** Когда агент сделает не то, что вы ожидали, это единственный способ понять, на каком шаге он свернул не туда. Ведите его с самого начала, а не добавляйте, когда что-то сломается.

---
## Шаг 6. Три способа никогда не остановиться

Цикл, который может не остановиться, — главная опасность этого урока. Способов три, и каждый закрывается отдельно.

| Способ не остановиться | Что происходит | Чем закрыт |
|---|---|---|
| Модель просит вызовы бесконечно | каждый шаг стоит денег | предел числа шагов |
| Модель повторяет один и тот же вызов | результат тот же, прогресса нет | сравнение с предыдущим вызовом |
| Инструмент падает, модель пробует снова | ошибка не мешает циклу крутиться | шаг тратится в любом случае |

Проверим все три.

*Статус ячейки: проверено запуском.*

In [ ]:
# 1. Модель просит и просит.
seq = [0]
def decide_endless(task, trace):
    seq[0] += 1
    return {"tool": "weather_forecast", "args": {"location": "М", "num_days": seq[0]}}

res = run_agent(decide_endless, reg, "x", max_steps=3)
print("1. Бесконечные вызовы  ->", res["stopped"], f"({len(res['trace'])} шага)")

# 2. Один и тот же вызов подряд.
def decide_loop(task, trace):
    return {"tool": "weather_forecast", "args": {"location": "М", "num_days": 1}}

res = run_agent(decide_loop, reg, "x", max_steps=10)
print("2. Повтор того же     ->", res["stopped"], f"({len(res['trace'])} шага из 10 разрешённых)")

# 3. Ошибка инструмента не останавливает цикл, но и не даёт крутиться вечно.
def decide_recover(task, trace):
    if not trace:
        return {"tool": "weather_forecast", "args": {"location": "М"}}      # нет num_days
    if len(trace) == 1:
        return {"tool": "weather_forecast", "args": {"location": "М", "num_days": 5}}
    return {"answer": "готово"}

res = run_agent(decide_recover, reg, "x", max_steps=5)
print("3. Ошибка и восстановление ->", res["stopped"])
for s in res["trace"]:
    print(f"     {s.n}. {s.kind}", f"— {s.error[:60]}" if s.error else "")

print()
try:
    run_agent(decide_normal, reg, "x", max_steps=0)
except ValueError as e:
    print("[ok] max_steps=0 отвергается:", e)

**Второй случай стоит рассмотреть внимательно.** Разрешено было десять шагов, а агент остановился на втором — потому что повтор того же вызова с теми же аргументами означает, что прогресса не будет. Восемь оставшихся шагов сэкономлены.

Без этой проверки предел шагов всё равно сработал бы, но вы бы заплатили за десять обращений вместо двух.

---
## Шаг 7. Настоящая модель

Единственный шаг с ключом. Вместо заглушки `decide` подставляем модель: передаём ей описания инструментов и разбираем то, что она вернёт.

Здесь используется модуль из урока 2.3: ответ модели о вызове функции — это тот же структурированный вывод, только со своим смыслом.

Что смотреть: выберет ли модель правильный инструмент, сгенерирует ли аргументы по схеме, не выдумает ли лишних.

*Статус ячейки: требует проверки на живом ключе. Автор материалов эту ячейку с настоящим ключом не запускал.*

In [ ]:
from llmcourse.structured import parse, StructureError

HAVE_KEY = not config.settings().get("OFFLINE", True)

SYSTEM = """Ты помощник, у которого есть инструменты.
Получив задачу, реши, нужен ли инструмент.

Если нужен — верни ТОЛЬКО JSON:
{{"tool": "имя", "args": {{...}}}}

Если можешь ответить сам — верни ТОЛЬКО JSON:
{{"answer": "текст ответа"}}

Доступные инструменты:
{tools}"""

if not HAVE_KEY:
    print("Ключ не подключён — шаг пропускается.")
    print("Шаги 0-6 дают всё содержание урока.")
else:
    import json as _j
    llm = client.LLM()
    system = SYSTEM.replace("{{", "{").replace("}}", "}").replace(
        "{tools}", _j.dumps(reg.describe(), ensure_ascii=False, indent=2))

    def decide_live(task, trace):
        """Роль decide играет модель."""
        history = ""
        if trace:
            done = [f"вызов {s.tool}({s.args}) -> {s.result}" if s.kind == "call"
                    else f"ошибка {s.tool}: {s.error}" for s in trace]
            history = "\n\nУже сделано:\n" + "\n".join(done)
        answer = llm.ask(task + history, system=system, temperature=0.1, max_tokens=300)
        try:
            return parse(answer)
        except StructureError as e:
            return {"answer": f"[не удалось разобрать ответ модели: {e}]"}

    res = run_agent(decide_live, reg, "Какая погода в Манжероке на 10 дней?",
                    max_steps=4)
    print("Остановился:", res["stopped"])
    print("Ответ:      ", res["answer"])
    print()
    for s in res["trace"]:
        print(f"  {s.n}. {s.kind} {s.tool} {s.args}")
        if s.error:
            print(f"      {s.error}")
        elif s.result is not None:
            print(f"      -> {s.result}")

---
## Задание

1. **Свой инструмент.** Опишите и зарегистрируйте инструмент, который считает количество рабочих дней между двумя датами. Продумайте описание: по нему модель должна отличить его от прогноза погоды.

2. **Плохое описание.** Замените описание погодного инструмента на «Возвращает данные» и посмотрите на шаге 7, что выберет модель. Опишите результат письменно — это и есть ответ на вопрос, зачем нужны описания.

3. **Разделение по последствиям.** Пройдите по своим инструментам и проставьте `writes`. Для каждого изменяющего напишите, что произойдёт в худшем случае при неверном вызове.

### Повышенной сложности

4. Добавьте в цикл агента подсчёт стоимости всех обращений (модуль из урока 2.2). Оцените, во что обходится агент, которому нужно в среднем три шага.

5. Сделайте так, чтобы изменяющая операция требовала подтверждения: агент останавливается, показывает, что собирается сделать, и продолжает только после явного разрешения.

6. Добавьте обнаружение зацикливания не только для соседних шагов, но и для повторов через один — модель может чередовать два вызова.

---
## Чек-лист

- [ ] Могу объяснить, почему модель не выполняет функции сама
- [ ] Описание инструмента написано так, что по нему можно сделать выбор
- [ ] Имя инструмента проверяется по белому списку до вызова
- [ ] Аргументы проверяются, включая случай выдуманного моделью аргумента
- [ ] Изменяющие операции отделены от читающих и запрещены по умолчанию
- [ ] Цикл агента ограничен по числу шагов и останавливается на повторе
- [ ] Журнал ведётся с самого начала, а не добавлен после первой поломки

## Частые проблемы

| Симптом | Причина и что делать |
|---|---|
| `ToolError: не зарегистрирован` | Модель назвала несуществующий инструмент. Проверьте описания: возможно, она не нашла подходящего и придумала |
| `ToolError: не описан в схеме` | Модель выдумала аргумент. Обычно помогает уточнить описания полей в схеме |
| `ToolError: изменяет состояние` | Инструмент помечен `writes`. Если это осознанно — передайте `allow_writes=True` |
| Агент останавливается на «зацикливание» | Модель повторяет один вызов. Значит, результата первого вызова ей не хватило — посмотрите, что именно возвращает инструмент |
| Агент упирается в предел шагов | Задача либо не решается имеющимися инструментами, либо описания не позволяют выбрать нужный |
| Модель выбирает не тот инструмент | Дело почти всегда в описаниях. Они должны различаться по смыслу, а не по названию |

## Что дальше

Урок 2.5 — **локальный запуск моделей**. Всё, что вы написали за четыре урока, будет работать с моделью на вашей машине: сменятся базовый адрес и имя модели, а код останется прежним. Это и есть та переносимость, ради которой в уроке 2.1 выбирался OpenAI-совместимый интерфейс.